### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [12]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = True

### Start with our Message class

In [13]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [14]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [15]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [16]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [17]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [18]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [19]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [20]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are several reasons in favor of choosing AutoGen for your AI Agent project:

1. **Modular Architecture**: AutoGen provides a modular framework that separates perception, reasoning, and action, facilitating easier development and scalability of applications.

2. **Complexity Handling**: It is designed to manage complex problems that may be too intricate for a single AI model, allowing for a more robust solution to various challenges.

3. **Multi-Agent Collaboration**: AutoGen excels in scenarios requiring collaboration among multiple agents, streamlining interactions and improving overall efficiency.

4. **Natural Language Handoffs**: The system simplifies coordination through natural language interactions, reducing the need for custom inter-agent protocols.

5. **Integration with Microsoft Ecosystem**: AutoGen seamlessly integrates with Azure AI services and other Microsoft tools, providing powerful capabilities for enterprise solutions.

6. **Code Generation and Debugging**: It supports autonomous code generation, allowing agents to write and execute code, as well as perform automated debugging, enhancing productivity.

7. **Customizability and Versatility**: AutoGen agents can be tailored to fit specific needs and operate in various modes that leverage combinations of LLMs, human inputs, and tools, promoting flexibility.

Overall, AutoGen offers a comprehensive set of features that can significantly enhance the development of complex AI-driven solutions. 

TERMINATE

## Cons of AutoGen:
Here are some cons associated with using AutoGen in an AI Agent project:

1. **Steep Learning Curve**: Users often face challenges when starting with AutoGen due to its complex features and functionalities, which may require significant time and effort to master.

2. **User Difficulty**: The interface and experience may not be user-friendly for everyone, particularly those who are not technically inclined, making it harder for teams to adopt.

3. **Lack of Intuitive Design**: Users may find the platform and its tools less intuitive, which can complicate the design and deployment of AI agents.

4. **Limited Visual Tools**: AutoGen lacks a comprehensive visual builder or no-code editor, which could hinder rapid prototyping and the ability to quickly iterate on designs.

5. **Fewer Integrations**: Compared to competitors like LangChain, AutoGen may offer a smaller ecosystem with fewer native integrations, potentially limiting functionality and flexibility in some use cases.

Overall, while AutoGen has powerful capabilities, these limitations may make it a less favorable choice, especially for teams prioritizing ease of use and rapid development. 

TERMINATE



## Decision:

Based on the research provided by the team, I recommend choosing AutoGen for the project. The strengths of AutoGen, particularly its modular architecture, ability to handle complexity through multi-agent collaboration, and integration with the Microsoft ecosystem, outweigh the challenges associated with its steep learning curve and less intuitive design. The advanced features like autonomous code generation and natural language handoffs can significantly boost productivity and innovation in developing complex AI-driven solutions. Although the user experience may pose some initial barriers, the long-term benefits and scalability offered by AutoGen present a compelling case for its adoption.

TERMINATE

In [21]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [22]:
await host.stop()